In [ ]:
!pip install pycm

In [ ]:
!pip install umap-learn

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import pycm
import scipy.optimize
import scipy.stats
import seaborn
import sklearn.cluster
import sklearn.datasets
import sklearn.decomposition
import sklearn.manifold
import sklearn.metrics
import tensorflow
import tensorboard
import torch
import torch.utils.tensorboard
import umap


# tensorflow.io.gfile = tensorboard.compat.tensorflow_stub.io.gfile

# Clustering & projections linéaires et non linéaires

## Chargement du dataset

Depuis sklearn : https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html

In [ ]:
digits = sklearn.datasets.load_digits()
print(digits.data.shape)
print(digits.target.shape)
seaborn.countplot(x=digits.target)

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(digits.data.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = digits.data[img_index]
    label = digits.target[img_index]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision par
    # ordinateur
    ax[i, j].imshow(image.reshape([8,8]), cmap='gray_r')
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Apprentissage

Définissez la fonction `train` qui prend en paramètre une matrice de features (`X`) et retourne un modèle KMeans à 10 clusters entraîné en utilisant la méthode `fit` d'une instance d'objet [sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html?highlight=kmeans#sklearn.cluster.KMeans).

In [ ]:
# Votre code ici

### Solution

In [ ]:
def train(data: numpy.ndarray) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=10).fit(data)

kmeans = train(digits.data)

## Utilisation du modèle appris pour labéliser chaque point

Utilisez la méthode `predict` pour labéliser chaque point du dataset.

In [ ]:
# Votre code ici

### Solution

In [ ]:
clusters = kmeans.predict(digits.data)

# On peut aussi directement utiliser labels_ pour récupérer les clusters des
# données d'entraînement
clusters = kmeans.labels_

print(clusters[:10])

## Affichage

Affichez les images correspondant à chaque centre de cluster en utilisant la méthode [`plt.imshow`](https://matplotlib.org/stable/plot_types/arrays/imshow.html#sphx-glr-plot-types-arrays-imshow-py).

Le centre d'un cluster est la centroïde calculé pour chaque cluster et stocké dans `kmeans.cluster_centers_`.

In [ ]:
plt.imshow([[1] * 7,
            [0] * 7,
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 1, 1, 0, 0],
            [0] * 7],
          cmap=plt.cm.binary)
plt.show()

# Votre code ici

### Solution

In [ ]:
fig, ax = plt.subplots(1, 10, figsize=(8, 3))

centers = kmeans.cluster_centers_.reshape(-1, 8, 8)

for axi, center in zip(ax, centers):
    axi.set(xticks=[], yticks=[])
    axi.imshow(center, cmap=plt.cm.binary)


## Assigner la classe majoritaire de chaque cluster

La fonction [`scipy.stats.mode`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mode.html) permet de calculer efficacement le mode de chaque cluster.

Définissez une fonction `predict` qui prend en arguments un modèle KMeans et une matrice de features et qui retourne le mode (dans les targets) du cluster de chaque point.

Nous avons aussi vu qu'il est possible d'utiliser l'algorithme hongrois pour calculer l'assignment des clusters aux labels. Utilisez [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html#scipy.optimize.linear_sum_assignment) et [`sklearn.metrics.cluster.contingency_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cluster.contingency_matrix.html) pour déployer cette méthode.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def predict_mode(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  clusters = model.labels_
  predictions = numpy.zeros_like(clusters)
  for i in range(10):
      mask = (clusters == i)
      predictions[mask] = scipy.stats.mode(digits.target[mask])[0]
  return predictions


# Même fonction en appliquant le pattern « split, apply, combine »
def predict_mode2(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  return (pandas.Series(digits.target)
                .groupby(model.labels_)
                .transform(lambda x: x.mode()[0])
                .values)


def predict_hungarian(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  clusters = model.labels_
  contmat = sklearn.metrics.cluster.contingency_matrix(digits.target, clusters)
  row_inds, col_inds = scipy.optimize.linear_sum_assignment(contmat, True)
  return col_inds.argsort()[clusters]


predict = predict_hungarian


predictions = predict(kmeans)

## Évaluation

Pour des prédictions données, calculez l'accuracy à l'aide des targets et de la fonction [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) de sklearn.

Affichez aussi la matrice de confusion entre les labels et les clusters. Au lieu d'utiliser la matrice de confusion de sklearn, utilisez le paquet [`pycm`](https://www.pycm.ir/doc/index.html) qui est plus abouti.

Enfin, calculez les scores de [silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) et l'[indice de Rand ajusté](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html) des clusters obtenus.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def evaluate(data: numpy.ndarray,
             predictions: numpy.ndarray,
             clusters: numpy.ndarray
             ) -> None:
  accuracy = sklearn.metrics.accuracy_score(digits.target, predictions)
  silhouette = sklearn.metrics.silhouette_score(data, clusters)
  rand_score = sklearn.metrics.cluster.adjusted_rand_score(digits.target,
                                                           clusters)
  print(f"Accuracy: {accuracy:.2f}, silhouette : {silhouette:.2f}, "
        f"rand_score : {rand_score:.2f}")
  mat = pycm.ConfusionMatrix(actual_vector=digits.target,
                             predict_vector=predictions)
  mat.plot()
  plt.show()
  if data.shape[1] == 2:
    _, (ax_left, ax_middle, ax_right) = plt.subplots(1, 3, figsize=(12, 4))
    ax_left.scatter(data[:, 0], data[:, 1], c=digits.target, alpha=0.7)
    ax_left.set_title("Classes")
    ax_left.axis("off")
    ax_middle.scatter(data[:, 0], data[:, 1], c=predictions, alpha=0.7)
    ax_middle.set_title("Prédictions")
    ax_middle.axis("off")
    ax_right.scatter(data[:, 0], data[:, 1], c=clusters, alpha=0.7)
    ax_right.set_title("Clusters")
    ax_right.axis("off")
    plt.show()


evaluate(digits.data, predictions, kmeans.labels_)

## Pipeline

Combinez les fonctions `train`, `predict` et `evaluate` pour définir une fonction `pipeline` qui prend en paramètre une matrice de features et qui affiche l'évaluation d'un KMeans entraîné dessus.

Appliquez cette pipeline :

- aux données originales
- aux données redécrites par PCA
- aux données redécrites par UMAP
- aux données redécrites par t-SNE

In [ ]:
# Votre code ici

### Solution

In [ ]:
def pipeline(data):
  model = train(data)
  clusters = model.labels_
  predictions = predict(train(data))
  evaluate(data, predictions, clusters)


print("Données originales")
pipeline(digits.data)
print()

print("PCA")
pca = sklearn.decomposition.PCA(n_components=0.8)
pca.fit(digits.data)
print("variance expliquée :", pca.explained_variance_ratio_)
print("variance cumulée   :", numpy.cumsum(pca.explained_variance_ratio_))
pipeline(pca.transform(digits.data))
print()

print("UMAP")
pipeline(umap.UMAP().fit_transform(digits.data))

print("t-SNE")
pipeline(sklearn.manifold.TSNE().fit_transform(digits.data))

## Visualisation avec TensorBoard Projector

L'outil TensorBoard est une merveille pour visualiser les données d'entraînement de tous types.

Nous allons voir ici comment l'utiliser pour visualiser chaque point par son image originale et sa target.

Une fois le tensorboard lancé, sélectionnez "Projector" dans le menu déroulant en haut à droite. Vous aurez alors une représentation des images dans un espace à 3 dimensions.

Afin de faciliter la lecture, on pourra selectionner "color by label" dans le menu à gauche.

**Attention** : un problème connu sur Tensorboard executé dans collaboratory peut rendre l'execution de T-SNE et UMAP incorrecte. L'ennui c'est que l'ensemble des facteurs menant à ce résultat est difficile à identifier... Si vous observez un nuage en forme de boule qui ne s'éclate pas au fil des itérations, demandez à voir une execution correcte.


In [ ]:
!rm -rf runs

vectors = numpy.array(digits.data)
metadata = digits.target  # labels
images = torch.LongTensor(digits.data.reshape(-1, 1, 8, 8))
writer = torch.utils.tensorboard.SummaryWriter()
writer.add_embedding(vectors, metadata, label_img=images)
writer.close()
%reload_ext tensorboard
%tensorboard --logdir=runs